# hidroxmx-forecasting — Colab entrypoint

Runs one stage of the Paper 2 pipeline on Google Colab with a time-limited GPU.

The notebook (i) clones this repository, (ii) installs the extras that need a GPU, (iii) reads R2 credentials from the Colab environment or from `google.colab.userdata`, (iv) restores `last.ckpt` for the requested `run-id` from R2, (v) runs exactly one script from `scripts/`, (vi) flushes new checkpoints and metrics to R2 and (vii) prints an exact resume command.

If the GPU session times out, re-open the notebook and re-run: the entrypoint reads the last checkpoint from R2 and continues within `checkpoint_every_n_steps` of where it stopped.

In [ ]:
# 1) Clone the repository (the notebook itself may already live inside it — re-clones only when running fresh in Colab).
%%bash --no-raise-error
if [ ! -d /content/hidroxmx-forecasting ]; then
    git clone --depth 1 https://github.com/pantrok/hidroxmx-forecasting.git /content/hidroxmx-forecasting
fi
cd /content/hidroxmx-forecasting && git rev-parse --short HEAD

In [ ]:
# 2) Install this repo with GPU extras (torch already ships with Colab).
!pip install -q -e /content/hidroxmx-forecasting[geo,uq]

In [ ]:
# 3) Read R2 credentials from Colab secrets or environment.
import os
try:
    from google.colab import userdata
    for k in ('R2_ENDPOINT_URL', 'R2_BUCKET', 'AWS_ACCESS_KEY_ID', 'AWS_SECRET_ACCESS_KEY',
              'R2_DATASET_PREFIX', 'R2_PAPER2_PREFIX'):
        try:
            os.environ[k] = userdata.get(k) or os.environ.get(k, '')
        except Exception:
            pass
except Exception:
    pass
assert os.environ.get('R2_ENDPOINT_URL'), 'set R2_ENDPOINT_URL and friends in Colab secrets or the current environment'

In [ ]:
# 4) Choose the stage and the run-id. The notebook does not train several
#    stages in a row — it drives exactly one, so a GPU timeout is bounded.
STAGE   = '11_train_forecaster'   # any script under scripts/, without .py
RUN_ID  = 'F1-cutzamala-2026-07'  # stable per experiment (donor set, seed…)
EXTRA_ARGS = '--physics --resume'
print(f'>>> stage: {STAGE}   run-id: {RUN_ID}')

In [ ]:
# 5) Restore last.ckpt for RUN_ID from R2 (if any), then run one stage.
!cd /content/hidroxmx-forecasting && \
 python scripts/{STAGE}.py --run-id {RUN_ID} {EXTRA_ARGS}

In [ ]:
# 6) Print an exact resume command in case the session ends before completion.
print('Resume command (paste after reconnect):')
print(f'  python scripts/{STAGE}.py --run-id {RUN_ID} {EXTRA_ARGS}')